# Notebook 09 — Zanbil.ir Web Server Log Evaluation

## Objective

Evaluate the NB07 model on a second independent, publicly available real-world web server log dataset from a completely different geographic and linguistic context.

**Dataset:** Web Server Access Logs — Zanbil.ir (Iranian e-commerce)  
**Source:** Harvard Dataverse DOI 10.7910/DVN/3QBYB5 (Zaker, Farzin, 2019)  
**Size:** 3.3GB — 10,365,152 lines  
**Language context:** Persian/Farsi e-commerce

## Machine Configuration

- CPU: Intel i7-1165G7 @ 2.80GHz (4 cores / 8 threads)
- RAM: 15.8GB usable
- Strategy: Sequential batch scoring (BATCH_SIZE=5000) with real-time progress

## Research Questions

> 1. What is the false positive rate on real Persian e-commerce traffic?
> 2. Does the dataset contain any real attacks?
> 3. How do Persian/Farsi query parameter values affect model scores?

**We do not assume this dataset is benign.** AIT-LDS-v1.1 was described as benign simulation traffic yet contained 1,156+ real attacks.

## 1. Imports & Setup

In [1]:
import os, re, csv, time, urllib.parse, warnings, json, sys
import numpy as np
import pandas as pd
import joblib
from collections import Counter
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

for d in ['results/models', 'results/figures', 'results/metrics']:
    os.makedirs(d, exist_ok=True)

# ── Config ────────────────────────────────────────────────────────
LOG_PATH     = '../logs/iranian_ecommerce_access.log'
OUTPUT_PATH  = 'results/metrics/09_results_checkpoint.csv'
MODEL_PATH   = 'results/models/07_rf_model.pkl'
VEC_PATH     = 'results/models/07_vectorizer.pkl'

T_HIGH       = 1.0
T_LOW        = 0.85
BATCH_SIZE   = 5000      # larger batch = more efficient predict_proba calls
TOTAL_LINES  = 10_365_152
REPORT_EVERY = 100_000   # print progress every 100K lines

FILE_SIZE = os.path.getsize(LOG_PATH)

print(f'Log file     : {LOG_PATH}')
print(f'File size    : {FILE_SIZE/1024**3:.2f} GB')
print(f'Total lines  : {TOTAL_LINES:,}')
print(f'Batch size   : {BATCH_SIZE:,}')
print(f'Report every : {REPORT_EVERY:,} lines')
print()
print('Setup complete.')

Log file     : ../logs/iranian_ecommerce_access.log
File size    : 3.26 GB
Total lines  : 10,365,152
Batch size   : 5,000
Report every : 100,000 lines

Setup complete.


## 2. Core Functions

In [2]:
SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ \[[^\]]+\] '
    r'"(?:\S+) (?P<url>.+?) HTTP/\d\.\d" '
    r'(?P<status>\d{3}) (?P<bytes>\S+)'
    r'(?: "[^"]*" "[^"]*")?'
)

def extract_query_values(url):
    try:
        parsed = urllib.parse.urlparse(url)
        params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
        values = [
            urllib.parse.unquote(v).strip()
            for vlist in params.values()
            for v in vlist
            if urllib.parse.unquote(v).strip()
        ]
        return ' '.join(values) if values else None
    except Exception:
        return None

def build_symbol_matrix(queries):
    rows = []
    for q in queries:
        c = Counter()
        for sym in SYMBOLS:
            c[sym] = str(q).count(sym)
        rows.append([c[sym] for sym in SYMBOLS])
    return csr_matrix(np.array(rows, dtype=float))

print('Core functions defined.')

Core functions defined.


## 3. Verify Log Format & Model

In [3]:
# Check first 5 lines
print('First 5 lines:')
print('-' * 80)
parsed_ok = 0
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.rstrip()[:120])
        if LOG_PATTERN.match(line.strip()):
            parsed_ok += 1

print(f'\nParser OK on {parsed_ok}/5 lines')
print()

# Check model
rf  = joblib.load(MODEL_PATH)
vec = joblib.load(VEC_PATH)
print(f'Model  : {type(rf).__name__}')
print(f'Vocab  : {len(vec.vocabulary_):,} features')

# Quick sanity score
test = ["1' UNION SELECT username,password FROM users--", "laptop", "جستجو"]
ngram = vec.transform(test)
sym   = build_symbol_matrix(test)
probs = rf.predict_proba(hstack([ngram, sym]))[:, 1]
print()
print('Sanity check:')
for t, p in zip(test, probs):
    print(f'  {p:.4f}  {t}')

del rf, vec, ngram, sym, probs
print('\nVerification complete.')

First 5 lines:
--------------------------------------------------------------------------------
54.36.149.41 - - [22/Jan/2019:03:56:14 +0330] "GET /filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,27|%
31.56.96.51 - - [22/Jan/2019:03:56:16 +0330] "GET /image/60844/productModel/200x200 HTTP/1.1" 200 5667 "https://www.zanb
31.56.96.51 - - [22/Jan/2019:03:56:16 +0330] "GET /image/61474/productModel/200x200 HTTP/1.1" 200 5379 "https://www.zanb
40.77.167.129 - - [22/Jan/2019:03:56:17 +0330] "GET /image/14925/productModel/100x100 HTTP/1.1" 200 1696 "-" "Mozilla/5.
91.99.72.15 - - [22/Jan/2019:03:56:17 +0330] "GET /product/31893/62100/%D8%B3%D8%B4%D9%88%D8%A7%D8%B1-%D8%AE%D8%A7%D9%86

Parser OK on 5/5 lines

Model  : RandomForestClassifier
Vocab  : 8,807 features

Sanity check:
  1.0000  1' UNION SELECT username,password FROM users--
  0.0400  laptop
  0.6840  جستجو

Verification complete.


## 4. Evaluate — Full File with Progress Tracking

Sequential batch scoring with real-time progress.

**Expected time: 60–90 minutes for 10.3M lines.**

Progress prints every 100,000 lines showing:
- Lines processed / total
- Current detection counts
- Elapsed time
- Estimated time remaining

In [4]:
# Load model once
print('Loading model...')
model = joblib.load(MODEL_PATH)
vec   = joblib.load(VEC_PATH)
print(f'Model loaded. Starting evaluation at {time.strftime("%H:%M:%S")}')
print(f'Batch size   : {BATCH_SIZE:,}')
print(f'Total lines  : {TOTAL_LINES:,}')
print()
print(f'{"Lines":>12}  {"Progress":>8}  {"ATTACK":>7}  {"SUSP":>6}  {"Elapsed":>8}  {"Remaining":>10}')
print('-' * 70)

t0 = time.perf_counter()

count_total  = 0
count_no_qs  = 0
count_scored = 0
count_attack = 0
count_susp   = 0
count_benign = 0
parse_errors = 0

buf_rows = []
buf_qvs  = []

def flush(writer):
    global count_attack, count_susp, count_benign, count_scored
    if not buf_qvs:
        return
    ngram = vec.transform(buf_qvs)
    sym   = build_symbol_matrix(buf_qvs)
    probs = model.predict_proba(hstack([ngram, sym]))[:, 1]
    for row, prob in zip(buf_rows, probs):
        prob = round(float(prob), 6)
        if prob >= T_HIGH:
            tier = 'ATTACK';     count_attack += 1
        elif prob >= T_LOW:
            tier = 'SUSPICIOUS'; count_susp   += 1
        else:
            tier = 'BENIGN';     count_benign += 1
        count_scored += 1
        writer.writerow({
            'url':    row[0],
            'qv':     row[1],
            'score':  prob,
            'tier':   tier,
            'status': row[2],
            'ip':     row[3],
        })
    buf_rows.clear()
    buf_qvs.clear()

with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=['url', 'qv', 'score', 'tier', 'status', 'ip']
    )
    writer.writeheader()

    with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        for raw_line in f:
            m = LOG_PATTERN.match(raw_line.strip())
            if not m:
                parse_errors += 1
                continue

            url    = m.group('url')
            status = int(m.group('status'))
            ip     = m.group('ip')
            qv     = extract_query_values(url)
            count_total += 1

            if qv is None:
                count_no_qs += 1
            else:
                buf_rows.append((url, qv, status, ip))
                buf_qvs.append(qv)
                if len(buf_qvs) >= BATCH_SIZE:
                    flush(writer)

            if count_total % REPORT_EVERY == 0:
                elapsed   = time.perf_counter() - t0
                rate      = count_total / elapsed
                remaining = (TOTAL_LINES - count_total) / rate if rate > 0 else 0
                pct       = count_total / TOTAL_LINES * 100
                print(
                    f'{count_total:>12,}  {pct:>7.1f}%  '
                    f'{count_attack:>7,}  {count_susp:>6,}  '
                    f'{elapsed/60:>7.1f}m  ~{remaining/60:>8.0f}m',
                    flush=True
                )

    flush(writer)  # final batch

elapsed = time.perf_counter() - t0
print()
print(f'Finished at  : {time.strftime("%H:%M:%S")}')
print(f'Total time   : {elapsed:.1f}s ({elapsed/60:.1f} minutes)')
print()
print(f'Total lines  : {count_total:,}')
print(f'Parse errors : {parse_errors:,}')
print(f'NO_QS        : {count_no_qs:,}')
print(f'Scored       : {count_scored:,}')
print(f'ATTACK       : {count_attack:,}')
print(f'SUSPICIOUS   : {count_susp:,}')
print(f'BENIGN       : {count_benign:,}')
print(f'Saved        : {OUTPUT_PATH}')

Loading model...
Model loaded. Starting evaluation at 10:16:23
Batch size   : 5,000
Total lines  : 10,365,152

       Lines  Progress   ATTACK    SUSP   Elapsed   Remaining
----------------------------------------------------------------------
     100,000      1.0%        0      14      0.0m  ~       4m
     200,000      1.9%        0      29      0.1m  ~       4m
     300,000      2.9%        0      55      0.1m  ~       4m
     400,000      3.9%        0      72      0.2m  ~       4m
     500,000      4.8%        0      83      0.2m  ~       4m
     600,000      5.8%        0      95      0.3m  ~       4m
     700,000      6.8%        0     123      0.3m  ~       4m
     800,000      7.7%        0     145      0.4m  ~       5m
     900,000      8.7%        0     159      0.4m  ~       5m
   1,000,000      9.6%        0     167      0.5m  ~       5m
   1,100,000     10.6%        0     179      0.6m  ~       5m
   1,200,000     11.6%        0     211      0.7m  ~       5m
   1,300,000

## 5. Results Analysis

In [5]:
total = count_total

fp_per_10k_attack = (count_attack / total) * 10000
fp_per_10k_susp   = (count_susp   / total) * 10000
fp_per_10k_all    = ((count_attack + count_susp) / total) * 10000

print('=' * 65)
print('ZANBIL.IR EVALUATION — NB07 MODEL')
print('=' * 65)
print(f'Dataset         : Zanbil.ir (Iranian e-commerce, 3.3GB)')
print(f'Total entries   : {total:,}')
print(f'Scored (has QS) : {count_scored:,} ({count_scored/total*100:.1f}%)')
print(f'NO_QS (skipped) : {count_no_qs:,} ({count_no_qs/total*100:.1f}%)')
print(f'Parse errors    : {parse_errors:,}')
print()
print(f'ATTACK  tier    : {count_attack:,}  ({fp_per_10k_attack:.2f} per 10,000)')
print(f'SUSPICIOUS tier : {count_susp:,}   ({fp_per_10k_susp:.2f} per 10,000)')
print(f'BENIGN          : {count_benign:,}')
print()
print(f'Combined FP/10k : {fp_per_10k_all:.2f}  (pending manual inspection)')
print()
print('INDEPENDENT DATASET PROGRESSION:')
print(f'  NB05 — Same env ATTACK FP/10k    :  0.25')
print(f'  NB08 — AIT-LDS (Austrian mail)   :  0.00  (verified)')
print(f'  NB09 — Zanbil (Iranian ecommerce): {fp_per_10k_attack:.2f}  (pending inspection)')

ZANBIL.IR EVALUATION — NB07 MODEL
Dataset         : Zanbil.ir (Iranian e-commerce, 3.3GB)
Total entries   : 10,365,075
Scored (has QS) : 1,874,503 (18.1%)
NO_QS (skipped) : 8,490,572 (81.9%)
Parse errors    : 77

ATTACK  tier    : 1  (0.00 per 10,000)
SUSPICIOUS tier : 2,779   (2.68 per 10,000)
BENIGN          : 1,871,723

Combined FP/10k : 2.68  (pending manual inspection)

INDEPENDENT DATASET PROGRESSION:
  NB05 — Same env ATTACK FP/10k    :  0.25
  NB08 — AIT-LDS (Austrian mail)   :  0.00  (verified)
  NB09 — Zanbil (Iranian ecommerce): 0.00  (pending inspection)


## 6. Inspect Detections

Do not assume these are false positives — manually inspect each one as per NB08 methodology.

In [6]:
import pandas as pd

detections_list = []
for chunk in pd.read_csv(OUTPUT_PATH, chunksize=5000):
    det = chunk[chunk['tier'].isin(['ATTACK', 'SUSPICIOUS'])]
    if len(det) > 0:
        detections_list.append(det)
    del chunk

if detections_list:
    det_df = pd.concat(detections_list).sort_values('score', ascending=False).reset_index(drop=True)
else:
    det_df = pd.DataFrame()

print(f'Total detections : {len(det_df)}')
print(f'  ATTACK         : {len(det_df[det_df["tier"]=="ATTACK"])}')
print(f'  SUSPICIOUS     : {len(det_df[det_df["tier"]=="SUSPICIOUS"])}')
print()

def categorise(qv):
    if not qv or str(qv) == 'nan':
        return 'Unknown'
    q = str(qv).lower()
    if 'union' in q and ('select' in q or 'all' in q):
        return 'SQLi — UNION'
    if "' or" in q or "1=1" in q or "or 1" in q:
        return 'SQLi — Tautology'
    if 'sleep(' in q or 'pg_sleep' in q or 'waitfor' in q or 'benchmark(' in q:
        return 'SQLi — Time-based'
    if 'select' in q and 'from' in q:
        return 'SQLi — Statement'
    if "'" in q and ('--' in q or '#' in q or '/*' in q):
        return 'SQLi — Quote+Comment'
    if '../' in q or 'etc/passwd' in q or 'boot.ini' in q:
        return 'Path Traversal'
    if 'script' in q or 'alert(' in q or 'javascript:' in q:
        return 'XSS'
    if 'system(' in q or 'exec(' in q or 'passthru' in q:
        return 'RCE'
    if "'" in q or ';' in q or '--' in q:
        return 'SQLi — Probe'
    return 'Unknown/Benign'

if len(det_df) > 0:
    det_df['attack_type'] = det_df['qv'].apply(categorise)
    print('Attack type breakdown:')
    print(det_df['attack_type'].value_counts().to_string())
    print()
    print(f'Top 20 detections by score:')
    print(f'{"Tier":<12} {"Score":>8} {"Type":<22} Query Values')
    print('-' * 95)
    for _, row in det_df.head(20).iterrows():
        qv = str(row['qv'])[:45] if row['qv'] else '-'
        print(f"{row['tier']:<12} {row['score']:>8.4f} {row['attack_type']:<22} {qv}")
    if len(det_df) > 20:
        print(f'\n... and {len(det_df)-20} more in 09_detections.csv')
    det_df.to_csv('results/metrics/09_detections.csv', index=False)
    print('Saved: results/metrics/09_detections.csv')
else:
    print('No detections found.')

Total detections : 2780
  ATTACK         : 1
  SUSPICIOUS     : 2779

Attack type breakdown:
attack_type
Unknown/Benign          1017
SQLi — Probe             981
SQLi — UNION             500
SQLi — Time-based        138
SQLi — Statement          95
SQLi — Tautology          33
SQLi — Quote+Comment      16

Top 20 detections by score:
Tier            Score Type                   Query Values
-----------------------------------------------------------------------------------------------
ATTACK         1.0000 Unknown/Benign         کمک‌بارفیکس (دستیاره) تن‌زیب
SUSPICIOUS     0.9900 SQLi — UNION           6aba3c.jpg&amp;wh=200x200' UNION ALL SELECT N
SUSPICIOUS     0.9800 SQLi — Statement       productModel 50x50') AND 3253=(SELECT UPPER(X
SUSPICIOUS     0.9800 SQLi — Statement       productModel 50x50 AND 3253=(SELECT UPPER(XML
SUSPICIOUS     0.9800 SQLi — Statement       productModel 50x50) AND 3253=(SELECT UPPER(XM
SUSPICIOUS     0.9800 SQLi — Statement       productModel 50x50) AND 32

## 7. High-Scoring BENIGN — Missed Attack Analysis

In [7]:
import pandas as pd

suspicious_benign = []
for chunk in pd.read_csv(OUTPUT_PATH, chunksize=5000):
    high = chunk[(chunk['tier'] == 'BENIGN') & (chunk['score'] >= 0.70)]
    if len(high) > 0:
        suspicious_benign.append(high)
    del chunk

result = pd.concat(suspicious_benign).sort_values('score', ascending=False).reset_index(drop=True) \
    if suspicious_benign else pd.DataFrame()

print(f'BENIGN entries scoring >= 0.70: {len(result)}')

if len(result) > 0:
    result['attack_type'] = result['qv'].apply(categorise)
    print('\nBreakdown:')
    print(result['attack_type'].value_counts().to_string())

    confirmed  = len(result[result['attack_type'] != 'Unknown/Benign'])
    likely_ben = len(result[result['attack_type'] == 'Unknown/Benign'])
    print(f'\nPotential missed attacks : {confirmed}')
    print(f'Likely genuine benign    : {likely_ben}')
    print()
    print(f'Top 20 by score:')
    print(f'{"Score":>8} {"Type":<22} Query Values')
    print('-' * 75)
    for _, row in result.head(20).iterrows():
        qv = str(row['qv'])[:45] if row['qv'] else '-'
        print(f"{row['score']:>8.4f} {row['attack_type']:<22} {qv}")
    if len(result) > 20:
        print(f'\n... and {len(result)-20} more in 09_high_scoring_benign.csv')

    result.to_csv('results/metrics/09_high_scoring_benign.csv', index=False)
    print('Saved: results/metrics/09_high_scoring_benign.csv')

BENIGN entries scoring >= 0.70: 220654

Breakdown:
attack_type
SQLi — Probe        193931
Unknown/Benign       26719
SQLi — Tautology         2
XSS                      2

Potential missed attacks : 193935
Likely genuine benign    : 26719

Top 20 by score:
   Score Type                   Query Values
---------------------------------------------------------------------------
  0.8400 Unknown/Benign         /Image/3(2).jpg
  0.8400 SQLi — Probe           41715_1145172065_ماشین-ظرفشویی-ال-جی-de24w---
  0.8400 Unknown/Benign         /Image/0(47).jpg
  0.8400 SQLi — Probe           41715_1145172065_ماشین-ظرفشویی-ال-جی-de24w---
  0.8400 Unknown/Benign         /Image/4(2).jpg
  0.8400 Unknown/Benign         /Image/4(13).jpg
  0.8400 Unknown/Benign         /Image/5(37).jpg
  0.8400 Unknown/Benign         /Image/3(11).jpg
  0.8400 Unknown/Benign         /Image/2(43).jpg
  0.8400 Unknown/Benign         /Image/4(2).jpg
  0.8400 Unknown/Benign         /Image/3(2).jpg
  0.8400 Unknown/Benign      

## 8. Persian/Farsi Query Analysis

In [8]:
import pandas as pd

def has_persian(text):
    if not text or str(text) == 'nan':
        return False
    return any('\u0600' <= c <= '\u06ff' for c in str(text))

benign_sample = []
for chunk in pd.read_csv(OUTPUT_PATH, chunksize=5000):
    b = chunk[chunk['tier'] == 'BENIGN'].head(50)
    if len(b) > 0:
        benign_sample.append(b)
    del chunk
    if len(benign_sample) >= 5:
        break

sample_df = pd.concat(benign_sample).head(200).reset_index(drop=True)
sample_df['has_persian'] = sample_df['qv'].apply(has_persian)

persian = sample_df[sample_df['has_persian']]
latin   = sample_df[~sample_df['has_persian']]

print(f'Sample of {len(sample_df)} BENIGN entries:')
print(f'  With Persian text  : {len(persian)}')
print(f'  Without Persian    : {len(latin)}')
print()

if len(persian) > 0:
    print('Persian score stats:')
    print(f'  Mean : {persian["score"].mean():.4f}')
    print(f'  Max  : {persian["score"].max():.4f}')
    print(f'  Min  : {persian["score"].min():.4f}')
    print()
    print('Sample Persian query values:')
    for _, row in persian.head(5).iterrows():
        print(f'  score={row["score"]:.4f}  {str(row["qv"])[:70]}')
    print()

if len(latin) > 0:
    print('Latin/numeric score stats:')
    print(f'  Mean : {latin["score"].mean():.4f}')
    print(f'  Max  : {latin["score"].max():.4f}')
    print(f'  Min  : {latin["score"].min():.4f}')

Sample of 200 BENIGN entries:
  With Persian text  : 6
  Without Persian    : 194

Persian score stats:
  Mean : 0.3950
  Max  : 0.7300
  Min  : 0.2400

Sample Persian query values:
  score=0.3900  فر-توکار-آشپزخانه-مدل-mf-0014-e.jpg max
  score=0.4800  p7,450|18 تا 28,v1|سفید
  score=0.2400  p0 لاوا لامپ
  score=0.2400  p0 لاوا لامپ
  score=0.7300  16 eshop.Order [{'name':'trackingCode','width':110},{'name':'ownerName

Latin/numeric score stats:
  Mean : 0.2692
  Max  : 0.7733
  Min  : 0.0000


## 9. Save Summary

In [9]:
import json

summary = {
    'dataset':             'Zanbil.ir Web Server Logs (Iranian e-commerce)',
    'source':              'Harvard Dataverse DOI 10.7910/DVN/3QBYB5',
    'total_entries':       count_total,
    'parse_errors':        parse_errors,
    'scored_entries':      count_scored,
    'no_qs_entries':       count_no_qs,
    'attack_tier':         count_attack,
    'suspicious_tier':     count_susp,
    'benign':              count_benign,
    'fp_per_10k_attack':   round((count_attack / count_total) * 10000, 4),
    'fp_per_10k_susp':     round((count_susp   / count_total) * 10000, 4),
    'fp_per_10k_combined': round(((count_attack + count_susp) / count_total) * 10000, 4),
    'note':                'Detections require manual inspection to confirm attack vs FP',
    'model':               'NB07 RF (07_rf_model.pkl)',
    'T_HIGH':              T_HIGH,
    'T_LOW':               T_LOW,
    'processing_minutes':  round(elapsed / 60, 1),
    'batch_size':          BATCH_SIZE,
}

with open('results/metrics/09_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved: results/metrics/09_summary.json')
print(json.dumps(summary, indent=2))

Saved: results/metrics/09_summary.json
{
  "dataset": "Zanbil.ir Web Server Logs (Iranian e-commerce)",
  "source": "Harvard Dataverse DOI 10.7910/DVN/3QBYB5",
  "total_entries": 10365075,
  "parse_errors": 77,
  "scored_entries": 1874503,
  "no_qs_entries": 8490572,
  "attack_tier": 1,
  "suspicious_tier": 2779,
  "benign": 1871723,
  "fp_per_10k_attack": 0.001,
  "fp_per_10k_susp": 2.6811,
  "fp_per_10k_combined": 2.6821,
  "note": "Detections require manual inspection to confirm attack vs FP",
  "model": "NB07 RF (07_rf_model.pkl)",
  "T_HIGH": 1.0,
  "T_LOW": 0.85,
  "processing_minutes": 8.7,
  "batch_size": 5000
}


## 10. Final Summary

In [10]:
print('=' * 65)
print('NOTEBOOK 09 — COMPLETE')
print('Zanbil.ir Evaluation — NB07 Model')
print('=' * 65)
print()
print(f'Dataset         : Zanbil.ir (Iranian e-commerce, 3.3GB)')
print(f'Total entries   : {count_total:,}')
print(f'Scored entries  : {count_scored:,} ({count_scored/count_total*100:.1f}%)')
print(f'Processing time : {elapsed/60:.1f} minutes')
print()
print(f'ATTACK tier     : {count_attack:,}')
print(f'SUSPICIOUS tier : {count_susp:,}')
print(f'BENIGN          : {count_benign:,}')
print(f'ATTACK FP/10k   : {(count_attack/count_total)*10000:.4f}  (pending manual inspection)')
print()
print('INDEPENDENT DATASET PROGRESSION:')
print(f'  NB05 — Same environment              :  0.25 FP/10k')
print(f'  NB08 — AIT-LDS v1.1 (Austria)        :  0.00 FP/10k  (verified)')
print(f'  NB09 — Zanbil.ir (Iran)              : {(count_attack/count_total)*10000:.2f} FP/10k  (pending)')
print()
print('NEXT STEPS:')
print('  1. Manually inspect all ATTACK/SUSPICIOUS detections')
print('  2. Inspect high-scoring BENIGN entries for missed attacks')
print('  3. Update true FP/10k after inspection')
print('  4. Document Persian character impact on scoring')

NOTEBOOK 09 — COMPLETE
Zanbil.ir Evaluation — NB07 Model

Dataset         : Zanbil.ir (Iranian e-commerce, 3.3GB)
Total entries   : 10,365,075
Scored entries  : 1,874,503 (18.1%)
Processing time : 8.7 minutes

ATTACK tier     : 1
SUSPICIOUS tier : 2,779
BENIGN          : 1,871,723
ATTACK FP/10k   : 0.0010  (pending manual inspection)

INDEPENDENT DATASET PROGRESSION:
  NB05 — Same environment              :  0.25 FP/10k
  NB08 — AIT-LDS v1.1 (Austria)        :  0.00 FP/10k  (verified)
  NB09 — Zanbil.ir (Iran)              : 0.00 FP/10k  (pending)

NEXT STEPS:
  1. Manually inspect all ATTACK/SUSPICIOUS detections
  2. Inspect high-scoring BENIGN entries for missed attacks
  3. Update true FP/10k after inspection
  4. Document Persian character impact on scoring


In [11]:
import pandas as pd
df = pd.read_csv('results/metrics/09_detections.csv')
print(df.groupby('attack_type')['score'].describe().round(4))

                       count    mean     std   min     25%   50%    75%   max
attack_type                                                                  
SQLi — Probe           981.0  0.8813  0.0238  0.85  0.8700  0.87  0.890  0.98
SQLi — Quote+Comment    16.0  0.9225  0.0218  0.88  0.9175  0.92  0.935  0.95
SQLi — Statement        95.0  0.9607  0.0170  0.92  0.9500  0.96  0.980  0.98
SQLi — Tautology        33.0  0.8924  0.0288  0.86  0.8700  0.88  0.910  0.97
SQLi — Time-based      138.0  0.9175  0.0295  0.87  0.9000  0.91  0.930  0.98
SQLi — UNION           500.0  0.9474  0.0171  0.91  0.9300  0.95  0.960  0.99
Unknown/Benign        1017.0  0.8820  0.0385  0.85  0.8500  0.86  0.934  1.00
